# FalaIbmec

Disponibilizamos o FalaIbmec em uma interface Gradio com identidade visual azul com respostas e fontes consultadas, incluindo arquivo, página, abrangência, link e trecho.

A busca usa Serafim pela biblioteca Sentence Transformers. A geração usa Qwen2.5-7B-Instruct com quantização em 4 bits. Os modelos e a política de contexto correspondem aos usados na avaliação do notebook 05.

Execute as células em ordem no **Google Colab com GPU T4** e envie **base_rag_v3.zip** quando solicitado. Este notebook carrega sua própria base e seus modelos. A interface apresenta cada pergunta de forma independente e pede uma mensagem completa.

In [ ]:
%pip install -q "sentence-transformers==5.7.0" "transformers==5.20.2" "accelerate==1.15.0" "bitsandbytes==0.50.2" "gradio==6.31.0"

In [ ]:
from pathlib import Path
import torch

assert torch.cuda.is_available(), "Ative GPU T4 nas configurações do Colab."
print("GPU", torch.cuda.get_device_name(0))
print("PyTorch", torch.__version__)
from google.colab import files
arquivos_interface = files.upload()
assert len(arquivos_interface) == 1, "Envie somente o ZIP da base v3."
raiz = Path("/content/projeto-pln")

In [ ]:
from zipfile import ZipFile
from io import BytesIO
import csv
import hashlib
import numpy as np

arquivos_esperados = {
    "data/processed/trechos_v3.csv",
    "data/processed/embeddings_trechos_v3.npy",
    "data/processed/indice_embeddings_v3.csv",
    "data/processed/configuracao_trechos_v3.csv",
}
# Aceita também nomes de upload com sufixos como (1) ou (2).
conteudo_base = next(iter(arquivos_interface.values()))
with ZipFile(BytesIO(conteudo_base)) as pacote:
    assert set(pacote.namelist()) == arquivos_esperados
    assert pacote.testzip() is None
    pacote.extractall(raiz)

pasta_dados = raiz / "data" / "processed"

def ler_csv(caminho):
    with caminho.open(encoding="utf-8-sig", newline="") as arquivo:
        return list(csv.DictReader(arquivo, delimiter=";"))

trechos = ler_csv(pasta_dados / "trechos_v3.csv")
indice_embeddings = ler_csv(pasta_dados / "indice_embeddings_v3.csv")
configuracao_base = ler_csv(pasta_dados / "configuracao_trechos_v3.csv")[0]
embeddings = np.load(pasta_dados / "embeddings_trechos_v3.npy", allow_pickle=False)
assert len(trechos) == len(indice_embeddings) == embeddings.shape[0] == 280
assert embeddings.shape[1] == 768
assert len({t["id_trecho"] for t in trechos}) == len(trechos)
for linha, (trecho, item) in enumerate(zip(trechos, indice_embeddings)):
    assert int(item["linha"]) == linha
    assert item["id_trecho"] == trecho["id_trecho"]
    assert item["modelo"] == configuracao_base["modelo_embeddings"]
    assert item["sha256_texto"] == hashlib.sha256(trecho["texto_trecho"].encode()).hexdigest()
assert np.isfinite(embeddings).all()
assert np.allclose(np.linalg.norm(embeddings, axis=1), 1.0, atol=1e-5)
print("Base conferida", len(trechos), "trechos e matriz", embeddings.shape)

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

assert torch.cuda.is_available(), "Ative a GPU antes de continuar."

nome_modelo_busca = configuracao_base["modelo_embeddings"]
revisao_busca = configuracao_base["revisao_modelo"]

modelo_busca = SentenceTransformer(
    nome_modelo_busca,
    revision=(
        revisao_busca
        if revisao_busca != "Não identificada"
        else None
    ),
    device="cpu",
)

assert modelo_busca.get_embedding_dimension() == embeddings.shape[1]

nome_llm = "Qwen/Qwen2.5-7B-Instruct"

configuracao_4bits = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizador_llm = AutoTokenizer.from_pretrained(nome_llm)

modelo_llm = AutoModelForCausalLM.from_pretrained(
    nome_llm,
    quantization_config=configuracao_4bits,
    device_map={"": 0},
    dtype=torch.float16,
)

modelo_llm.eval()

print("Busca:", nome_modelo_busca)
print("Gerador:", nome_llm)
print("Quantização em 4 bits:", modelo_llm.is_loaded_in_4bit)
print("Dispositivo:", modelo_llm.get_input_embeddings().weight.device)

## Funções do RAG

Recuperamos os três primeiros candidatos e selecionamos os trechos do mesmo registro do primeiro resultado. O modelo gera a resposta usando esse contexto. O painel de fontes permite conferir os documentos efetivamente enviados ao modelo.

In [ ]:
def responder_rag(
    pergunta,
    trechos,
    embeddings,
    modelo_busca,
    modelo_llm,
    tokenizador_llm,
    k=3,
):
    import re
    import numpy as np
    import torch

    if not pergunta.strip():
        raise ValueError("Digite uma pergunta.")

    assert 1 <= k <= len(trechos)

    vetor = modelo_busca.encode(
        pergunta,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    pontuacoes = embeddings @ vetor
    indices = np.argsort(-pontuacoes)[:k]

    encontrados = [
        {
            **trechos[int(indice)],
            "similaridade": float(pontuacoes[indice]),
        }
        for indice in indices
    ]

    blocos = []

    for numero, trecho in enumerate(encontrados, start=1):
        metadados = [
            f"[{numero}] Arquivo: {trecho['arquivo']}",
            f"Trecho: {trecho['id_trecho']}",
        ]

        for campo, descricao in {
            "pagina": "Página",
            "mes": "Mês",
            "abrangencia_documentada": "Abrangência",
            "periodo_documentado": "Período",
            "limites_uso": "Limitações",
        }.items():
            if trecho.get(campo):
                metadados.append(f"{descricao}: {trecho[campo]}")

        blocos.append(
            "\n".join(metadados)
            + "\nConteúdo:\n"
            + trecho["texto_trecho"]
        )

    prompt = """
Responda à PERGUNTA em português brasileiro, de forma natural e breve,
usando exclusivamente as informações dos DOCUMENTOS.

Preserve os nomes dos menus e procedimentos.
Não invente etapas, documentos exigidos, botões ou explicações de siglas.
Respeite as limitações de abrangência e vigência das fontes.

Não escreva URLs nem marcadores de referência.
O sistema exibirá separadamente os documentos consultados.

Se os documentos não contiverem informação para responder, diga:
"Não encontrei informação suficiente nos documentos consultados."
""".strip()

    contexto = "\n\n".join(blocos)

    mensagens = [
        {"role": "system", "content": prompt},
        {
            "role": "user",
            "content": f"DOCUMENTOS:\n{contexto}\n\nPERGUNTA:\n{pergunta}",
        },
    ]

    entradas = tokenizador_llm.apply_chat_template(
        mensagens,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(modelo_llm.get_input_embeddings().weight.device)

    tamanho_entrada = entradas["input_ids"].shape[1]

    assert (
        tamanho_entrada + 384
        <= modelo_llm.config.max_position_embeddings
    ), "A entrada excede o contexto disponível."

    with torch.inference_mode():
        saida = modelo_llm.generate(
            **entradas,
            max_new_tokens=384,
            do_sample=False,
            pad_token_id=tokenizador_llm.eos_token_id,
        )

    resposta = tokenizador_llm.decode(
        saida[0, tamanho_entrada:],
        skip_special_tokens=True,
    ).strip()

    conteudo = re.sub(r"\[[\d,\s]+\]", "", resposta)
    tem_conteudo = bool(re.search(r"[A-Za-zÀ-ÿ]", conteudo))

    return {
        "pergunta": pergunta,
        "resposta": (
            resposta if tem_conteudo
            else "Não consegui gerar uma resposta válida."
        ),
        "saida_original": resposta,
        "fontes_consultadas": encontrados,
        "modo": "gerada" if tem_conteudo else "revisar",
    }


def responder_rag_focado(
    pergunta,
    trechos,
    embeddings,
    modelo_busca,
    modelo_llm,
    tokenizador_llm,
    k=3,
):
    import numpy as np

    if not pergunta.strip():
        raise ValueError("Digite uma pergunta.")

    assert 1 <= k <= len(trechos)

    vetor = modelo_busca.encode(
        pergunta,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    pontuacoes = embeddings @ vetor
    candidatos = np.argsort(-pontuacoes)[:k]

    registro_principal = trechos[
        int(candidatos[0])
    ]["id_registro"]

    selecionados = [
        int(indice)
        for indice in candidatos
        if trechos[int(indice)]["id_registro"] == registro_principal
    ]

    resultado = responder_rag(
        pergunta,
        [trechos[indice] for indice in selecionados],
        embeddings[selecionados],
        modelo_busca,
        modelo_llm,
        tokenizador_llm,
        k=len(selecionados),
    )

    resultado["politica_contexto"] = "mesmo_registro_do_primeiro_resultado"
    resultado["candidatos_recuperados"] = [
        {
            "id_trecho": trechos[int(indice)]["id_trecho"],
            "similaridade": float(pontuacoes[indice]),
        }
        for indice in candidatos
    ]

    return resultado


print("Funções do RAG preparadas.")

## Chatbot

Execute a próxima célula e use o campo de mensagem. Abra **De onde veio a informação?** para conferir a origem da informação. A interface fica disponível enquanto a sessão Colab estiver ativa.

In [ ]:
import gradio as gr
from html import escape
from urllib.parse import urlparse
from datetime import datetime, timezone

registros_interface = globals().get("registros_interface", [])


def formatar_fontes(fontes):
    blocos = []

    for numero, fonte in enumerate(fontes, start=1):
        titulo = fonte.get("titulo") or fonte["arquivo"]

        detalhes = [
            f"<h3>{numero}. {escape(titulo)}</h3>",
            f"<p>Arquivo: {escape(fonte['arquivo'])}</p>",
        ]

        if fonte.get("pagina"):
            detalhes.append(
                f"<p>Página: {escape(str(fonte['pagina']))}</p>"
            )

        if fonte.get("abrangencia_documentada"):
            detalhes.append(
                f"<p>Abrangência: {escape(fonte['abrangencia_documentada'])}</p>"
            )

        if fonte.get("periodo_documentado"):
            detalhes.append(
                f"<p>Período: {escape(fonte['periodo_documentado'])}</p>"
            )

        url = (
            fonte.get("url_documento")
            or fonte.get("url_referencia")
            or fonte.get("url")
        )

        if url and urlparse(url).scheme in {"http", "https"}:
            detalhes.append(
                f'<p><a href="{escape(url, quote=True)}" '
                'target="_blank" rel="noopener noreferrer">'
                "Abrir fonte</a></p>"
            )

        detalhes.append(
            '<blockquote style="white-space: pre-wrap;">'
            + escape(fonte["texto_trecho"])
            + "</blockquote>"
        )

        blocos.append("\n".join(detalhes))

    return "\n<hr>\n".join(blocos)


def conversar(mensagem, historico):
    resultado = responder_rag_focado(
        mensagem,
        trechos,
        embeddings,
        modelo_busca,
        modelo_llm,
        tokenizador_llm,
    )

    registros_interface.append({
        "data_hora": datetime.now(timezone.utc).isoformat(),
        **resultado,
    })

    return (
        resultado["resposta"],
        formatar_fontes(resultado["fontes_consultadas"]),
    )


CSS_FALAIBMEC = """
.gradio-container {
    box-sizing: border-box !important;
    width: 100% !important;
    max-width: 1140px !important;
    margin: 0 auto !important;
    padding: 28px 24px !important;
    background: #f3f7fc !important;
}
.gradio-container .main { padding: 0 !important; min-width: 0 !important; }
#fala-cabecalho { padding: 0; border: 0; background: transparent; }
.fala-capa {
    background: #082b59;
    color: #ffffff;
    border-radius: 24px;
    padding: 28px 34px 30px;
    overflow: hidden;
    position: relative;
}
.fala-marca { display: flex; align-items: center; gap: 12px; }
.fala-simbolo {
    display: grid; place-items: center;
    width: 62px; height: 62px; flex-shrink: 0;
    background: #1c64d8; border: 1px solid #79aeff;
    border-radius: 14px 14px 14px 3px;
    color: #ffffff; font-size: 22px; font-weight: 800; letter-spacing: -1px;
}
.fala-simbolo svg { display: block; width: 54px; height: 54px; }
.fala-identidade { color: #c8defa; font-size: 12px; letter-spacing: 1.8px; text-transform: uppercase; }
.fala-capa h1 {
    font-family: inherit; font-size: 40px; font-weight: 800;
    letter-spacing: -1.8px; line-height: 1.15;
    color: #ffffff; margin: 22px 0 10px;
}
.fala-capa h1 span { color: #83c6ff; }
.fala-capa p { margin: 0; color: #d1e4fb; font-size: 15px; line-height: 1.7; max-width: 620px; }
.fala-campus {
    position: absolute; right: 32px; top: 32px;
    padding: 7px 12px; border: 1px solid #466a98;
    border-radius: 30px; color: #deebff; font-size: 12px;
}
#fala-conteudo { gap: 20px; margin-top: 8px; align-items: flex-start; }
#fala-conversa {
    background: #ffffff; border: 1px solid #dce6f3;
    border-radius: 20px; padding: 18px;
    box-shadow: 0 10px 30px rgba(8, 43, 89, 0.035);
}
#fala-chat { border: none; background: #ffffff; outline: none; box-shadow: none; }
#fala-chat .message.user { background: #e7f0ff !important; color: #123d77 !important; }
#fala-chat .message.bot { background: #f5f8fc !important; color: #173354 !important; }
.fala-boas-vindas { max-width: 380px; margin: auto; text-align: center; padding: 35px 20px; }
.fala-boas-vindas strong { display: block; color: #123d77; font-size: 23px; margin-bottom: 10px; }
.fala-boas-vindas p { color: #63768f; font-size: 14px; line-height: 1.7; }
#fala-mensagem textarea { padding: 14px 16px !important; font-size: 14px; }
#fala-mensagem button { background: #185bc4 !important; color: #ffffff !important; border-radius: 12px; padding: 10px 16px; }
#fala-mensagem button:hover { background: #104ba7 !important; }
#fala-fontes { background: #ffffff; border: 1px solid #dce6f3; border-radius: 20px; }
#fala-fontes button { color: #173f73; font-weight: 700; }
#fala-fontes-conteudo { background: transparent; border: 0; }
#fala-fontes-conteudo h3 { font-size: 15px; color: #143d72; margin: 14px 0 10px; line-height: 1.5; }
#fala-fontes-conteudo p { font-size: 12px; line-height: 1.7; color: #62748c; margin: 4px 0; overflow-wrap: anywhere; }
#fala-fontes-conteudo a { display: inline-block; color: #1557ba; font-weight: 700; margin: 7px 0; }
#fala-fontes-conteudo blockquote {
    margin: 12px 0; border-left: 3px solid #7fb6ff;
    background: #f2f7ff; padding: 12px 14px; color: #354f70;
    font-size: 12px; line-height: 1.8; max-height: 250px; overflow: auto;
}
.fala-fontes-vazias { padding: 20px 6px; }
.fala-fontes-vazias strong { display: block; color: #294c76; font-size: 14px; margin-bottom: 10px; }
#fala-rodape { text-align: center; color: #6a7f98; font-size: 12px; padding-top: 6px; }
@media (max-width: 760px) {
    .gradio-container { padding: 14px 10px !important; }
    .fala-capa { padding: 22px; border-radius: 18px; }
    .fala-campus { position: static; display: inline-block; margin-top: 18px; }
    .fala-capa h1 { font-size: 33px; margin-top: 20px; }
    #fala-conteudo { flex-direction: column !important; align-items: stretch !important; }
    #fala-conversa, #fala-fontes-coluna {
        flex: none !important; width: 100% !important;
        min-width: 0 !important; box-sizing: border-box;
    }
    #fala-conversa { padding: 10px; }
}
"""

tema_falaibmec = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="sky",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Manrope"), "sans-serif"],
).set(
    body_background_fill="#f3f7fc",
    body_background_fill_dark="#f3f7fc",
    body_text_color="#173354",
    body_text_color_dark="#173354",
    body_text_color_subdued="#62748c",
    body_text_color_subdued_dark="#62748c",
    background_fill_primary="#ffffff",
    background_fill_primary_dark="#ffffff",
    background_fill_secondary="#f3f7fc",
    background_fill_secondary_dark="#f3f7fc",
    block_background_fill="#ffffff",
    block_background_fill_dark="#ffffff",
    block_border_color="#dce6f3",
    block_border_color_dark="#dce6f3",
    block_label_text_color="#294c76",
    block_label_text_color_dark="#294c76",
    block_label_background_fill="#ffffff",
    block_label_background_fill_dark="#ffffff",
    input_background_fill="#f7faff",
    input_background_fill_dark="#f7faff",
    input_border_color="#cbdcf1",
    input_border_color_dark="#cbdcf1",
    input_placeholder_color="#70839c",
    input_placeholder_color_dark="#70839c",
    button_primary_background_fill="#185bc4",
    button_primary_background_fill_dark="#185bc4",
    button_primary_background_fill_hover="#104ba7",
    button_primary_background_fill_hover_dark="#104ba7",
    button_primary_text_color="#ffffff",
    button_primary_text_color_dark="#ffffff",
    button_secondary_background_fill="#eef5ff",
    button_secondary_background_fill_dark="#eef5ff",
    button_secondary_text_color="#234e86",
    button_secondary_text_color_dark="#234e86",
    button_secondary_border_color="#d6e5fa",
    button_secondary_border_color_dark="#d6e5fa",
    block_radius="16px",
    input_radius="14px",
    button_large_radius="14px",
)

with gr.Blocks(title="FalaIbmec") as interface:
    gr.HTML(
        """
        <header class="fala-capa">
            <div class="fala-marca">
                <span class="fala-simbolo" aria-hidden="true">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 64 64" fill="none">
  <path d="M32 13V8" stroke="#b9e6ff" stroke-width="3" stroke-linecap="round"/>
  <circle cx="32" cy="6" r="3.5" fill="#83d7ff"/>
  <rect x="10" y="25" width="7" height="12" rx="3.5" fill="#83c6ff"/>
  <rect x="47" y="25" width="7" height="12" rx="3.5" fill="#83c6ff"/>
  <rect x="15" y="14" width="34" height="31" rx="12" fill="#f4fbff"/>
  <rect x="20" y="21" width="24" height="15" rx="7.5" fill="#0b3466"/>
  <path d="M25 28.5c1.2-2.2 3-2.2 4.2 0M35 28.5c1.2-2.2 3-2.2 4.2 0"
        stroke="#91e0ff" stroke-width="2.4" stroke-linecap="round"/>
  <path d="M28 39c2.5 2 5.5 2 8 0" stroke="#2665af" stroke-width="2.2" stroke-linecap="round"/>
  <circle cx="21" cy="36" r="2" fill="#b6deff"/>
  <circle cx="43" cy="36" r="2" fill="#b6deff"/>
  <path d="M25 45h14c5.5 0 10 4.5 10 10v4H15v-4c0-5.5 4.5-10 10-10Z" fill="#83c6ff"/>
  <path d="M24 49c3 0 5.6 1 8 3 2.4-2 5-3 8-3v9c-3 0-5.6 1-8 3-2.4-2-5-3-8-3v-9Z" fill="#f4fbff"/>
  <path d="M32 52v8" stroke="#2764a7" stroke-width="1.8" stroke-linecap="round"/>
</svg>
</span>
                <span class="fala-identidade">Assistente acadêmico</span>
            </div>
            <h1>Fala<span>Ibmec</span></h1>
            <p>Uma conversa para facilitar sua vida acadêmica.<br>
            Tire suas dúvidas e confira a informação nos documentos.</p>
            <span class="fala-campus">Ibmec · Faria Lima</span>
        </header>
        """,
        elem_id="fala-cabecalho",
    )

    with gr.Row(elem_id="fala-conteudo"):
        with gr.Column(scale=7, min_width=360, elem_id="fala-conversa") as coluna_conversa:
            conversa = gr.Chatbot(
                label="Conversa com o FalaIbmec",
                show_label=False,
                height=390,
                layout="bubble",
                buttons=["copy"],
                placeholder=(
                    '<div class="fala-boas-vindas">'
                    '<strong>Oi! Como posso ajudar?</strong>'
                    '<p>Pergunte sobre sua vida acadêmica ou escolha '
                    'um dos assuntos abaixo para começar.</p></div>'
                ),
                elem_id="fala-chat",
            )
            campo_mensagem = gr.Textbox(
                placeholder="Escreva sua pergunta aqui…",
                elem_id="fala-mensagem",
                show_label=False,
                container=False,
                lines=1,
                max_lines=4,
                submit_btn="Enviar",
                stop_btn="Parar",
            )
            # O painel é criado primeiro para ser ligado à saída adicional do chat.
        with gr.Column(scale=3, min_width=270, elem_id="fala-fontes-coluna"):
            with gr.Accordion("De onde veio a informação?", open=True, elem_id="fala-fontes"):
                painel_fontes = gr.HTML(
                    '<div class="fala-fontes-vazias">'
                    '<strong>Fontes da última resposta</strong>'
                    '<p>Depois da sua pergunta, os documentos consultados '
                    'aparecem aqui com página, link e trecho.</p></div>',
                    elem_id="fala-fontes-conteudo",
                )

    with coluna_conversa:
        chat = gr.ChatInterface(
            fn=conversar,
            chatbot=conversa,
            textbox=campo_mensagem,
            additional_outputs=[painel_fontes],
            examples=[
                "Como solicitar a carteirinha de estudante?",
                "Como é calculada a média final das disciplinas?",
                "Como enviar a documentação de estágio pelo SIA?",
            ],
            example_labels=["Carteirinha", "Média final", "Estágio"],
            examples_label="Experimente perguntar",
            run_examples_on_click=False,
            cache_examples=False,
            autofocus=False,
            concurrency_limit=1,
        )

    gr.Markdown(
        "Protótipo acadêmico · Faça uma pergunta completa por mensagem. "
        "Confira a abrangência e o período dos documentos consultados.",
        elem_id="fala-rodape",
    )

interface.queue(default_concurrency_limit=1).launch(
    share=True,
    inline=True,
    theme=tema_falaibmec,
    css=CSS_FALAIBMEC,
)


## Conclusão

Integramos a busca semântica e a geração de respostas à interface de conversa. O painel de fontes apresenta os documentos consultados e seus metadados para apoiar a conferência das respostas. As métricas do piloto estão no notebook 05.